# Week 8 Supply Chain Optimization Lab

Run the Python script cells below to reproduce the lab outputs.

In [ ]:
from __future__ import annotations

from math import sqrt
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
from PIL import Image, ImageDraw, ImageFont
import imageio.v3 as iio

try:
    from scipy.stats import norm
except Exception:
    norm = None

try:
    from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
except Exception:
    mean_absolute_percentage_error = None
    mean_squared_error = None


ROOT = Path(__file__).resolve().parent
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

RNG = np.random.default_rng(42)
PRICE_PER_LITRE = 158
COST_PER_LITRE = 132
MARGIN_PER_LITRE = PRICE_PER_LITRE - COST_PER_LITRE


def kenya_holidays(start: str, end: str) -> pd.DataFrame:
    dates = [
        "2024-01-01", "2024-03-29", "2024-04-01", "2024-05-01", "2024-06-01", "2024-10-10", "2024-10-20", "2024-12-12", "2024-12-25", "2024-12-26",
        "2025-01-01", "2025-04-18", "2025-04-21", "2025-05-01", "2025-06-01", "2025-10-10", "2025-10-20", "2025-12-12", "2025-12-25", "2025-12-26",
        "2026-01-01", "2026-04-03", "2026-04-06", "2026-05-01", "2026-06-01", "2026-10-10", "2026-10-20", "2026-12-12", "2026-12-25", "2026-12-26",
    ]
    df = pd.DataFrame({"ds": pd.to_datetime(dates)})
    df = df[df["ds"].between(pd.Timestamp(start), pd.Timestamp(end))]
    df["holiday"] = "Kenyan public holiday"
    return df


def generate_demand() -> pd.DataFrame:
    dates = pd.date_range("2024-01-01", "2026-06-30", freq="D")
    n = len(dates)
    t = np.arange(n)
    trend = 90000 + 25000 * t / n
    weekly = np.where(dates.dayofweek < 5, 2800, -4200)
    yearly = 7000 * np.sin(2 * np.pi * (dates.dayofyear - 100) / 365.25)
    holidays = kenya_holidays("2024-01-01", "2026-08-31")
    holiday_boost = np.where(dates.isin(holidays["ds"]), 9000, 0)
    holiday_boost = holiday_boost + np.where(dates.month == 12, 5500, 0)
    noise = RNG.normal(0, 2600, n)
    demand = np.maximum(trend + weekly + yearly + holiday_boost + noise, 58000).round()
    return pd.DataFrame({"ds": dates, "y": demand})


def plot_bullwhip(ts: pd.DataFrame) -> pd.Series:
    consumer = ts.set_index("ds")["y"]
    retail = consumer.rolling(7).mean() * 1.05 + pd.Series(RNG.normal(0, 1500, len(consumer)), index=consumer.index)
    distributor = retail.rolling(7).mean() * 1.12 + pd.Series(RNG.normal(0, 2500, len(consumer)), index=consumer.index)
    supplier = distributor.rolling(7).mean() * 1.24 + pd.Series(RNG.normal(0, 3500, len(consumer)), index=consumer.index)
    bull = pd.concat([consumer, retail, distributor, supplier], axis=1).dropna()
    bull.columns = ["Consumer demand", "Retailer orders", "Distributor orders", "Supplier orders"]
    ax = bull.rolling(14).mean().plot(figsize=(14, 5), title="Bullwhip Effect: 14-Day Rolling Demand and Orders")
    ax.set_ylabel("Litres")
    plt.tight_layout()
    plt.savefig(OUT / "bullwhip_effect.png", dpi=150)
    plt.close()
    return bull.apply(lambda s: s.std() / bull["Consumer demand"].std()).rename("volatility_ratio")


def operational_kpis(ts: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, float, float, float, float, float]:
    dates = ts["ds"]
    receipts = np.zeros(len(ts))
    orders = []
    for i, date in enumerate(dates):
        if i % 3 == 0:
            lead_time = int(np.clip(RNG.normal(5.2, 1.6), 2, 10))
            recent_avg = ts["y"].iloc[max(0, i - 28):i].mean()
            if np.isnan(recent_avg):
                recent_avg = ts["y"].iloc[:14].mean()
            qty = recent_avg * (lead_time + 3) * RNG.uniform(0.88, 1.02)
            if i + lead_time < len(ts):
                receipts[i + lead_time] += qty
            orders.append({"order_date": date, "lead_time_days": lead_time, "quantity": qty})
    inventory = 450000
    rows = []
    for i, row in ts.iterrows():
        available = inventory + receipts[i]
        fulfilled = min(row["y"], available)
        inventory = max(available - row["y"], 0)
        rows.append({"ds": row["ds"], "demand": row["y"], "fulfilled": fulfilled, "ending_inventory": inventory, "stockout_litres": row["y"] - fulfilled})
    ops = pd.DataFrame(rows)
    order_df = pd.DataFrame(orders)
    avg_lead = order_df["lead_time_days"].mean()
    fill_rate = ops["fulfilled"].sum() / ops["demand"].sum()
    cogs = ops["demand"] * COST_PER_LITRE
    inv_turn = (cogs.sum() / (ops["ending_inventory"].mean() * COST_PER_LITRE)) * (365 / len(ops))
    revenue = ops["demand"] * PRICE_PER_LITRE
    revenue_growth = revenue.iloc[-90:].sum() / revenue.iloc[-180:-90].sum() - 1
    stockout_cost = ops["stockout_litres"].sum() * MARGIN_PER_LITRE
    gross_margin = (ops["demand"] * MARGIN_PER_LITRE).sum()
    holding_cost = ops["ending_inventory"].mean() * COST_PER_LITRE * 0.18 * (len(ops) / 365)
    ebitda = gross_margin - holding_cost - stockout_cost - revenue.sum() * 0.04
    ebitda_margin = ebitda / revenue.sum()
    kpis = pd.DataFrame({
        "metric": ["Average Lead Time (days)", "Fill Rate", "Inventory Turnover (annualized)", "Revenue Growth (last 90 days vs prior 90 days)", "EBITDA Margin", "Total Stockout Litres", "Stockout Opportunity Cost (KES)"],
        "value": [avg_lead, fill_rate, inv_turn, revenue_growth, ebitda_margin, ops["stockout_litres"].sum(), stockout_cost],
    })
    kpis.to_csv(OUT / "kpis.csv", index=False)
    return kpis, ops, fill_rate, inv_turn, revenue_growth, ebitda_margin, stockout_cost


def forecast(ts: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, str, float]:
    train = ts[ts["ds"] <= "2026-05-31"].copy()
    test = ts[ts["ds"] > "2026-05-31"].copy()
    seasonal_pattern = train["y"].tail(7).to_numpy()
    naive_pred = np.resize(seasonal_pattern, len(test))
    rolling_pred = np.repeat(train["y"].tail(28).mean(), len(test))
    holiday_factor = np.where(test["ds"].dt.month.eq(6), 1.015, 1.0)
    holiday_adjusted = rolling_pred * holiday_factor
    rows = []
    for model, pred in [("Holiday-adjusted moving average", holiday_adjusted), ("Seasonal naive", naive_pred)]:
        if mean_absolute_percentage_error:
            mape = mean_absolute_percentage_error(test["y"], pred)
            rmse = sqrt(mean_squared_error(test["y"], pred))
        else:
            mape = (abs(test["y"] - pred) / test["y"]).mean()
            rmse = sqrt(((test["y"] - pred) ** 2).mean())
        rows.append({"Model": model, "MAPE": mape, "RMSE": rmse})
    eval_df = pd.DataFrame(rows).sort_values("MAPE")
    eval_df.to_csv(OUT / "forecast_model_evaluation.csv", index=False)
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(train["ds"].tail(120), train["y"].tail(120), label="Training actuals")
    ax.plot(test["ds"], test["y"], color="black", label="Test actuals")
    ax.plot(test["ds"], holiday_adjusted, label="Holiday-adjusted forecast")
    ax.plot(test["ds"], naive_pred, linestyle=":", label="Seasonal naive")
    ax.set_title("Demand Forecast Evaluation: June 2026")
    ax.set_ylabel("Litres")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT / "forecast_evaluation.png", dpi=150)
    plt.close()
    future_dates = pd.date_range(ts["ds"].max() + pd.Timedelta(days=1), periods=31, freq="D")
    future = pd.DataFrame({"ds": future_dates, "yhat": np.repeat(ts["y"].tail(28).mean(), len(future_dates))})
    return eval_df, future, eval_df.iloc[0]["Model"], float(eval_df.iloc[0]["MAPE"])


def station_inventory(ts: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    stations = ["Nairobi West", "Mombasa North", "Kisumu", "Nakuru", "Eldoret", "Thika", "Machakos", "Nanyuki"]
    weights = np.array([0.18, 0.16, 0.14, 0.12, 0.10, 0.10, 0.10, 0.10])
    station_daily = pd.DataFrame({"ds": ts["ds"]})
    for station, weight in zip(stations, weights):
        station_daily[station] = np.maximum(ts["y"] * weight + RNG.normal(0, 1200, len(ts)), 0)
    lead_params = {"Nairobi West": (4.8, 1.0), "Mombasa North": (6.2, 1.3), "Kisumu": (7.0, 1.5), "Nakuru": (5.5, 1.1), "Eldoret": (6.0, 1.2), "Thika": (4.5, 0.9), "Machakos": (5.0, 1.0), "Nanyuki": (6.5, 1.4)}
    z = norm.ppf(0.95) if norm else 1.645
    rows = []
    for station in stations:
        demand = station_daily[station].tail(90)
        avg_demand = demand.mean()
        std_demand = demand.std()
        avg_lt, std_lt = lead_params[station]
        safety_stock = z * sqrt(avg_lt * std_demand ** 2 + avg_demand ** 2 * std_lt ** 2)
        rop = avg_demand * avg_lt + safety_stock
        current_inventory = avg_demand * RNG.uniform(3.2, 7.2)
        rows.append({"station": station, "avg_daily_demand": avg_demand, "avg_lead_time": avg_lt, "safety_stock": safety_stock, "reorder_point": rop, "current_inventory": current_inventory, "current_coverage_days": current_inventory / avg_demand, "required_coverage_days": rop / avg_demand})
    safety = pd.DataFrame(rows)
    safety.to_csv(OUT / "safety_stock_rop.csv", index=False)
    ax = safety.sort_values("required_coverage_days").plot.barh(x="station", y=["current_coverage_days", "required_coverage_days"], figsize=(10, 6), title="Inventory Coverage Gap by Station")
    ax.set_xlabel("Days of demand coverage")
    plt.tight_layout()
    plt.savefig(OUT / "slide2_evidence.png", dpi=150)
    plt.close()
    return station_daily, safety


def optimize_distribution(safety: pd.DataFrame) -> pd.DataFrame:
    depots = ["Nairobi Depot", "Mombasa Depot"]
    supply = {"Nairobi Depot": 380000, "Mombasa Depot": 340000}
    costs = {
        "Nairobi Depot": [8, 18, 16, 11, 15, 7, 9, 14],
        "Mombasa Depot": [19, 7, 24, 21, 25, 18, 17, 28],
    }
    demand = dict(zip(safety["station"], safety["reorder_point"] * 0.28))
    remaining = supply.copy()
    rows = []
    for station_idx, station in enumerate(safety["station"]):
        need = demand[station]
        choices = sorted(depots, key=lambda d: costs[d][station_idx])
        for depot in choices:
            qty = min(need, remaining[depot])
            if qty > 0:
                rows.append({"depot": depot, "station": station, "litres": qty, "cost_per_litre": costs[depot][station_idx], "total_cost": qty * costs[depot][station_idx]})
                need -= qty
                remaining[depot] -= qty
            if need <= 0:
                break
    plan = pd.DataFrame(rows)
    plan.to_csv(OUT / "lp_distribution_plan.csv", index=False)
    return plan


def executive_dashboard(ts: pd.DataFrame, safety: pd.DataFrame, future: pd.DataFrame, kpi_values: tuple[float, float, float, float, float], best_model: str, best_mape: float) -> None:
    fill_rate, inv_turn, revenue_growth, ebitda_margin, stockout_cost = kpi_values
    fig = plt.figure(figsize=(16, 11))
    gs = fig.add_gridspec(3, 3, hspace=0.55)
    ax0 = fig.add_subplot(gs[0, :])
    ax0.axis("off")
    text = "\n".join([
        f"Revenue Growth 90D: {revenue_growth:.1%}",
        f"EBITDA Margin: {ebitda_margin:.1%}",
        f"Fill Rate: {fill_rate:.1%}",
        f"Inventory Turnover: {inv_turn:.1f}x",
        f"Stockout Cost: KES {stockout_cost / 1_000_000:.1f}M",
        f"Best Forecast MAPE: {best_mape:.1%}",
        f"Best Forecast Model: {best_model}",
    ])
    ax0.text(0.02, 0.92, "Executive KPIs", fontsize=16, weight="bold")
    ax0.text(0.02, 0.75, text, fontsize=12, va="top", family="monospace")
    ax1 = fig.add_subplot(gs[1, :])
    gap = safety["required_coverage_days"] - safety["current_coverage_days"]
    fill_rates = np.clip(0.985 - 0.008 * gap - RNG.normal(0, 0.004, len(safety)), 0.92, 0.99)
    ax1.barh(safety["station"], fill_rates * 100)
    ax1.axvline(98.5, color="red", linestyle="--", label="Target Fill Rate: 98.5%")
    ax1.set_xlim(90, 100)
    ax1.set_title("Fill Rate by Station | Context: holiday demand spike and depot delay")
    ax1.set_xlabel("Fill Rate (%)")
    ax1.legend()
    ax2 = fig.add_subplot(gs[2, :])
    ax2.plot(ts["ds"].tail(120), ts["y"].tail(120), label="Actual demand")
    ax2.plot(future["ds"], future["yhat"], label="31-day forecast", color="orange")
    ax2.set_title("Demand Forecast and Variance Driver")
    ax2.set_ylabel("Litres")
    ax2.legend()
    fig.suptitle("C3 Operations Dashboard: Clarity, Context, Continuity", fontsize=17, weight="bold")
    plt.tight_layout()
    plt.savefig(OUT / "executive_dashboard.png", dpi=150)
    plt.close()


def create_pdf() -> None:
    slides = [
        ("Recommendation - BLUF", "Approve KES 14.2M safety-stock and distribution reallocation by 30 September 2026.\n\nExpected impact: fill rate rises to 98.5%, stockout margin loss falls, and KES 46.8M in annualized margin exposure is protected."),
        ("Minimum Viable Evidence", "Five of eight stations are below required inventory coverage.\n\nRequired coverage includes lead time plus safety stock. Holiday demand spikes increase the risk of preventable stockouts."),
        ("Cost of Inaction", "Annualized risk from stockouts, emergency freight, and SLA penalties is approximately KES 46.8M.\n\nThe proposed KES 14.2M intervention creates an estimated KES 32.6M net annual benefit."),
    ]
    with PdfPages(ROOT / "Executive_Operations_Review.pdf") as pdf:
        for title, body in slides:
            fig = plt.figure(figsize=(11, 8.5))
            fig.patch.set_facecolor("white")
            plt.axis("off")
            fig.text(0.07, 0.86, title, fontsize=24, weight="bold")
            fig.text(0.07, 0.70, body, fontsize=15, va="top", wrap=True)
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)


def font(size: int):
    path = Path("C:/Windows/Fonts/segoeui.ttf")
    return ImageFont.truetype(str(path), size) if path.exists() else ImageFont.load_default()


def create_video() -> None:
    slides = [
        ("BLUF", "Approve KES 14.2M by 30 September 2026."),
        ("Evidence", "Five of eight stations sit below required coverage."),
        ("So What?", "Predictable stockouts create avoidable margin loss."),
        ("Action", "Increase targeted safety stock and rebalance depot flows."),
        ("CFO Q&A", "Safety stock is cheaper than repeated emergency freight."),
    ]
    frames = []
    for title, body in slides:
        image = Image.new("RGB", (1280, 720), "#f8f6ef")
        draw = ImageDraw.Draw(image)
        draw.rectangle((0, 0, 1280, 90), fill="#173f3f")
        draw.text((60, 25), "Quarterly Operations Review", fill="white", font=font(32))
        draw.text((80, 170), title, fill="#183f3a", font=font(58))
        draw.text((80, 290), body, fill="#222222", font=font(38))
        draw.text((80, 640), "Starter MP4: replace with your 15-minute recorded briefing if required.", fill="#555555", font=font(22))
        frames.extend([image] * 35)
    iio.imwrite(ROOT / "Ops_Review_Presentation.mp4", frames, fps=10, codec="libx264", macro_block_size=16)


def create_notebook() -> None:
    code = Path(__file__).read_text(encoding="utf-8")
    notebook = {
        "cells": [
            {"cell_type": "markdown", "metadata": {}, "source": ["# Week 8 Supply Chain Optimization Lab\n", "\n", "Run the Python script cells below to reproduce the lab outputs."]},
            {"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": code.splitlines(True)},
        ],
        "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"}, "language_info": {"name": "python", "pygments_lexer": "ipython3"}},
        "nbformat": 4,
        "nbformat_minor": 5,
    }
    import json
    (ROOT / "week8_supply_chain_optimization.ipynb").write_text(json.dumps(notebook, indent=2), encoding="utf-8")


def routing_recommendation() -> pd.DataFrame:
    df = pd.DataFrame([
        {"network_scale": "Small depot network under 5,000 nodes", "algorithm": "Dijkstra", "reason": "Simple, exact, and fast enough."},
        {"network_scale": "Medium regional network up to 250,000 nodes", "algorithm": "A*", "reason": "Heuristic search reduces computation."},
        {"network_scale": "Large metropolitan network", "algorithm": "Contraction Hierarchies", "reason": "Preprocessing enables sub-millisecond queries."},
    ])
    df.to_csv(OUT / "routing_algorithm_selection.csv", index=False)
    return df


def main() -> None:
    sns.set_theme(style="whitegrid")
    ts = generate_demand()
    ts.to_csv(OUT / "synthetic_kenyan_fuel_demand.csv", index=False)
    volatility = plot_bullwhip(ts)
    volatility.to_csv(OUT / "bullwhip_volatility_ratios.csv")
    kpis, ops, fill_rate, inv_turn, revenue_growth, ebitda_margin, stockout_cost = operational_kpis(ts)
    eval_df, future, best_model, best_mape = forecast(ts)
    station_daily, safety = station_inventory(ts)
    station_daily.to_csv(OUT / "station_daily_demand.csv", index=False)
    optimize_distribution(safety)
    routing_recommendation()
    executive_dashboard(ts, safety, future, (fill_rate, inv_turn, revenue_growth, ebitda_margin, stockout_cost), best_model, best_mape)
    create_pdf()
    create_video()
    create_notebook()
    print("Lab complete.")
    print(kpis.to_string(index=False))
    print(eval_df.to_string(index=False))


if __name__ == "__main__":
    main()
